# calculation of distance module with astropy and sncosmo

- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/CNRS
- creation date : 2026-05-09

`documentation:`
- https://sncosmo.readthedocs.io/en/stable/index.html
- https://lsstdesc.org/SN-PWV/notebooks/simulating_lc_for_cadence.html#The-PLAsTICC-Data

In [ ]:
import sncosmo
from astropy.cosmology import FlatLambdaCDM
import astropy.units as u

import numpy as np
import matplotlib.pyplot as plt

## Constants

In [ ]:
#ZP = 31.4
#ZP = 17.5
ZP = 25
bandprefix = "lsst"
bandsnames = ["u","g","r","i","z","y"]
bandscolors = ["b","g","r","orange","grey","k"]
t = np.linspace(-20, 50, 100)  # Temps en jours autour du pic
NB = len(bandsnames)

## Function for module de distance

In [ ]:
def mu_theo_vs_z(z, cosmo=None):
    """
    Calcule μ théorique vs z avec astropy.

    Args:
        z (float ou array): Redshift(s).
        cosmo: Cosmologie astropy (par défaut: FlatLambdaCDM avec H0=70, Om0=0.3).

    Returns:
        mu_theo (float ou array): Module de distance théorique.
    """
    if cosmo is None:
        cosmo = FlatLambdaCDM(H0=70 * u.km / u.s / u.Mpc, Om0=0.3)
    return cosmo.distmod(z).value

In [ ]:
def mu_vs_z_salt2(z, cosmo=None, alpha=0.14, beta=3.1, M=-19.3, zp=ZP,b="lsstr"):
    if cosmo is None:
        cosmo = FlatLambdaCDM(H0=70 * u.km / u.s / u.Mpc, Om0=0.3)

    model = sncosmo.Model(source='salt2')
    model.set(z=z, t0=0, x0=1.0, x1=0.0, c=0.0)  # x1 et c à 0 pour la référence

    t = np.linspace(-20, 50, 100)
    flux_b = model.bandflux(b, t, zpsys='flux')  # Flux en Jy
    flux_b = np.array(flux_b)  # Convertir en array si nécessaire

    # Convertir le flux en magnitude (système AB)
    m_b = -2.5 * np.log10(flux_b) + zp
    m_b_pic = np.min(m_b)  # Magnitude au pic (minimum car plus brillant = magnitude plus faible)

    # Calcul de μ
    mu = m_b_pic - M + alpha * model.get('x1') - beta * model.get('c')
    return mu

In [ ]:
def mu_vs_z_salt3(z, cosmo=None, alpha=0.14, beta=3.1, M=-19.3, zp=ZP,b="lsstr"):
    if cosmo is None:
        cosmo = FlatLambdaCDM(H0=70 * u.km / u.s / u.Mpc, Om0=0.3)

    model = sncosmo.Model(source='salt3')
    model.set(z=z, t0=0, x0=1.0, x1=0.0, c=0.0)

    t = np.linspace(-20, 50, 100)
    flux_b = model.bandflux(b, t, zpsys='flux')  # Flux en Jy
    flux_b = np.array(flux_b)

    # Convertir en magnitude
    m_b = -2.5 * np.log10(flux_b) + zp
    m_b_pic = np.min(m_b)

    mu = m_b_pic - M + alpha * model.get('x1') - beta * model.get('c')
    return mu

In [ ]:
def mu_vs_z_salt3_coherent(z, cosmo=None, alpha=0.14, beta=3.1, M=-19.3, zp=ZP ,b="lsstr"):
    if cosmo is None:
        cosmo = FlatLambdaCDM(H0=70 * u.km / u.s / u.Mpc, Om0=0.3)

    # Calculer μ théorique
    mu_theo = cosmo.distmod(z).value

    # Calculer m_B (magnitude apparente au pic)
    x1 = 0.0  # Stretch
    c = 0.0   # Couleur
    m_B = mu_theo + M - alpha * x1 + beta * c

    # Initialiser le modèle SALT3
    model = sncosmo.Model(source='salt3')
    model.set(z=z, t0=0, x0=1.0, x1=x1, c=c)

    # Générer la courbe de lumière en bande g
    t = np.linspace(-20, 50, 100)
    flux_b = model.bandflux(b, t, zpsys='flux')  # Flux en Jy

    # Normaliser le flux pour correspondre à m_B
    flux_b_normalized = flux_b * 10**(-0.4 * (m_B - zp))

    # Convertir en magnitude
    m_b = -2.5 * np.log10(flux_b_normalized) + zp
    m_b_pic = np.min(m_b)

    # Calculer μ empirique (doit correspondre à μ_théo)
    mu_emp = m_b_pic - M + alpha * x1 - beta * c
    return mu_emp

## 1. Définition du module de distance $\mu$
Le module de distance est une quantité centrale en cosmologie, surtout pour les supernovae de type Ia. Il relie directement la distance de luminosité $d_L$​ (en parsecs) à la différence entre la magnitude apparente mmm (observée) et la magnitude absolue M (intrinsèque) :
$$
\mu = m - M = 5 \log_{10}\left(\frac{d_L}{10 \text{ pc}}\right)
$$

Utilité :

Permet de standardiser les luminosités des supernovae (après correction de la forme de la courbe de lumière et de la couleur).
Facilite la comparaison directe entre les magnitudes observées et les modèles théoriques.
Est utilisé pour contraindre les paramètres cosmologiques (ex : $H_0$​, $\Omega_m$​,$\Omega_\Lambda$
).


## Calculation

In [ ]:
z_values = np.linspace(0.01, 1.5, 100)

In [ ]:
mu_values_astropy = [mu_theo_vs_z(z) for z in z_values]

In [ ]:
mu_values_salt2 = [ mu_vs_z_salt2(z,zp=ZP ,b="lsstr") for z in z_values]

In [ ]:
mu_values_salt3 = [ mu_vs_z_salt3(z,zp=ZP ,b="lsstr") for z in z_values]
mu_values_salt3_coherent = [ mu_vs_z_salt3_coherent(z,zp=ZP ,b="lsstr") for z in z_values]

In [ ]:
print("z\tμ (SALT2)\tμ (SALT3)\tμ (SALT3-coh)\tμ (théorique)")
for z, m2, m3, m3c, mt in zip(z_values, mu_values_salt2, mu_values_salt3,mu_values_salt3_coherent, mu_values_astropy):
    print(f"{z:.2f}\t{m2:.2f}\t\t{m3:.2f}\t\t{m3c:.2f}\t\t{mt:.2f}")

## Plot

In [ ]:
fig,ax = plt.subplots(1,1)

ax.plot(z_values , mu_values_astropy,c="k" ,label = "astropy")
ax.plot(z_values , mu_values_salt2, c="b",label = "salt2")
ax.plot(z_values , mu_values_salt3, c="r",label = "salt3")
ax.plot(z_values , mu_values_salt3_coherent, c="purple",label = "salt3-coherent")
ax.legend()
ax.set_xlabel("redshift")
ax.set_ylabel("distance modulus in mag")
ax.set_title("distance modulus in mag vs redshift")